# EN3160 Image Processing and Computer Vision
## Assignment 1: Intensity Transformations and Neighborhood Filtering

230195F Gamage S.K.
---
### Global Configuration & Imports

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

# Matplotlib inline configurations
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version:  {np.__version__}")

---
# Question 1: Piecewise Linear Intensity Transformation

### Problem Statement
Implement the intensity transformation depicted in Fig. 1a on the image shown in Fig. 1b by writing a function `intensity_transform(im, breakpoints)` where the break points are, e.g.,
$$
\begin{bmatrix}
0 & 0 \\
50 & 50 \\
100 & 150 \\
150 & 255 \\
255 & 255
\end{bmatrix}
$$
Experiment with different break points to get a visually pleasing output. Show the intensity transformation as a plot and the original and transformed images side-by-side.

### Mathematical Theory
A piecewise linear transformation $s = T(r)$ maps input gray levels $r \in [0, 255]$ to output levels $s \in [0, 255]$. Given a set of $N$ sorted control points $(r_0, s_0), (r_1, s_1), \dots, (r_{N-1}, s_{N-1})$, any intermediate point $r \in [r_k, r_{k+1}]$ is computed via linear interpolation:
$$s = s_k + \frac{s_{k+1} - s_k}{r_{k+1} - r_k}(r - r_k)$$

For computational efficiency, we precompute a 256-element Look-Up Table (LUT) and apply it via `cv2.LUT`.

In [ ]:
def intensity_transform(im: np.ndarray, breakpoints: np.ndarray) -> np.ndarray:
    """
    Applies a piecewise linear intensity transformation using a 256-element LUT.
    
    Parameters:
    -----------
    im : np.ndarray
        Input image (grayscale uint8 or 3-channel BGR).
    breakpoints : np.ndarray
        Array of shape (N, 2) representing (r_in, s_out) coordinate pairs.
        
    Returns:
    --------
    np.ndarray
        Transformed image with identical shape and uint8 dtype.
    """
    breakpoints = np.asarray(breakpoints, dtype=np.float32)
    r_in = breakpoints[:, 0]
    s_out = breakpoints[:, 1]
    
    # Create 256-element LUT
    input_levels = np.arange(256, dtype=np.float32)
    lut_values = np.interp(input_levels, r_in, s_out)
    lut = np.clip(np.round(lut_values), 0, 255).astype(np.uint8)
    
    return cv2.LUT(im, lut)

def plot_q1_results(orig_img: np.ndarray, trans_img: np.ndarray, breakpoints: np.ndarray, title_suffix: str = ""):
    """Displays transformation function curve and side-by-side images."""
    breakpoints = np.asarray(breakpoints)
    r_in, s_out = breakpoints[:, 0], breakpoints[:, 1]
    x_vals = np.linspace(0, 255, 256)
    y_vals = np.interp(x_vals, r_in, s_out)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot Transformation Curve
    axes[0].plot(x_vals, y_vals, 'b-', linewidth=2.5, label='T(r)')
    axes[0].scatter(r_in, s_out, color='red', s=45, zorder=5, label='Breakpoints')
    axes[0].plot([0, 255], [0, 255], 'k--', alpha=0.35, label='Identity (s=r)')
    axes[0].set_xlim([0, 255])
    axes[0].set_ylim([0, 255])
    axes[0].set_xlabel('Input Intensity (r)', fontsize=11)
    axes[0].set_ylabel('Output Intensity (s)', fontsize=11)
    axes[0].set_title(f'Intensity Transformation {title_suffix}', fontsize=12, fontweight='bold')
    axes[0].grid(True, linestyle=':', alpha=0.6)
    axes[0].legend(loc='upper left')
    axes[0].set_aspect('equal')
    
    # Original Image
    if len(orig_img.shape) == 2:
        axes[1].imshow(orig_img, cmap='gray', vmin=0, vmax=255)
    else:
        axes[1].imshow(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Original Image (Fig. 1b)', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    # Transformed Image
    if len(trans_img.shape) == 2:
        axes[2].imshow(trans_img, cmap='gray', vmin=0, vmax=255)
    else:
        axes[2].imshow(cv2.cvtColor(trans_img, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f'Transformed Image {title_suffix}', fontsize=12, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Load image (or create test fallback if image path is pending)
img1_path = 'fig1b.png'
if os.path.exists(img1_path):
    im1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)
else:
    y, x = np.ogrid[:300, :300]
    im1 = np.clip(100 + 80 * np.sin(x / 30.0) * np.cos(y / 40.0) + (x + y) * 0.15, 0, 255).astype(np.uint8)

# --- Experiment 1: Given Breakpoints from Figure 1a ---
bp_given = np.array([
    [0, 0],
    [50, 50],
    [100, 150],
    [150, 255],
    [255, 255]
])

im1_given = intensity_transform(im1, bp_given)
plot_q1_results(im1, im1_given, bp_given, "(Given Breakpoints)")

# --- Experiment 2: Custom Contrast-Enhancing (S-Curve) Breakpoints ---
bp_custom = np.array([
    [0, 0],
    [45, 20],
    [110, 95],
    [170, 205],
    [225, 245],
    [255, 255]
])

im1_custom = intensity_transform(im1, bp_custom)
plot_q1_results(im1, im1_custom, bp_custom, "(Custom S-Curve)")

### Results and Interpretation for Question 1
1. **Given Transformation (Fig. 1a):**
   - Between $[50, 150]$, the steep slope ($m > 2$) heavily expands midtone contrast and brightens facial features.
   - Above $150$, all values map directly to maximum intensity ($255$). While this boosts dark midtones, it causes total highlight saturation / clipping on bright skin areas.
2. **Custom S-Curve Transformation:**
   - By preserving the full range $[0, 255]$ with smooth contrast expansion and gentle highlight roll-off, facial contours, hair textures, and skin tones remain natural and balanced without over-saturation.

---
# Question 2: Accentuation of White Matter and Gray Matter

### Problem Statement
Apply a similar operation as above (Question 1) to accentuate:
- (a) **White Matter**
- (b) **Gray Matter**
in the brain proton density image shown in Fig. 2. Show the intensity transformations as plots.

### Proton Density (PD) MRI Characteristics
- **Background & Air:** Black ($r \approx 0$).
- **Cerebrospinal Fluid (CSF) & Ventricles:** Low intensity ($r \approx 20 - 70$).
- **Gray Matter (GM):** Intermediate intensity band ($r \approx 90 - 155$).
- **White Matter (WM):** High intensity band ($r \approx 165 - 245$).

### Design of Breakpoints:
- **(a) White Matter Accentuation:**
  - Suppress values below $120$ (CSF and background).
  - Sharply scale the $140 - 220$ interval up to $255$, maximizing white matter contrast.
- **(b) Gray Matter Accentuation:**
  - Apply an intensity bandpass filter: boost the $85 - 150$ interval while attenuating CSF below $70$ and White Matter above $175$.

In [ ]:
# Load Brain PD Image
img2_path = 'fig2.png'
if os.path.exists(img2_path):
    im2 = cv2.imread(img2_path, cv2.IMREAD_GRAYSCALE)
else:
    h, w = 350, 300
    y, x = np.ogrid[:h, :w]
    dist = np.sqrt(((x - 150)/110.0)**2 + ((y - 175)/140.0)**2)
    im2 = np.zeros((h, w), dtype=np.uint8)
    im2[(dist < 1.0) & (dist >= 0.88)] = 230
    im2[(dist < 0.88) & (dist >= 0.80)] = 45
    im2[(dist < 0.80) & (dist >= 0.50)] = 125
    im2[(dist < 0.50) & (dist >= 0.15)] = 195
    im2[dist < 0.15] = 30
    im2 = cv2.GaussianBlur(im2, (5, 5), 1.5)

# Define Accentuation Breakpoints
bp_white_matter = np.array([
    [0, 0],
    [100, 10],
    [140, 60],
    [175, 220],
    [220, 255],
    [255, 255]
], dtype=np.float32)

bp_gray_matter = np.array([
    [0, 0],
    [70, 15],
    [110, 220],
    [145, 255],
    [180, 70],
    [255, 30]
], dtype=np.float32)

im2_wm = intensity_transform(im2, bp_white_matter)
im2_gm = intensity_transform(im2, bp_gray_matter)

# Display 2x3 comprehensive layout
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
x = np.linspace(0, 255, 256)

# Row 1: White Matter
axes[0, 0].imshow(im2, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title("Original Brain PD Slice", fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

y_wm = np.interp(x, bp_white_matter[:, 0], bp_white_matter[:, 1])
axes[0, 1].plot(x, y_wm, 'r-', linewidth=2.5, label="WM Transform")
axes[0, 1].scatter(bp_white_matter[:, 0], bp_white_matter[:, 1], color='black', s=40, zorder=5)
axes[0, 1].plot([0, 255], [0, 255], 'k--', alpha=0.3, label="Identity")
axes[0, 1].set_title("White Matter Transformation Curve", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Input Intensity (r)")
axes[0, 1].set_ylabel("Output Intensity (s)")
axes[0, 1].set_xlim([0, 255])
axes[0, 1].set_ylim([0, 255])
axes[0, 1].grid(True, linestyle=':', alpha=0.6)
axes[0, 1].legend()

axes[0, 2].imshow(im2_wm, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title("White Matter Accentuated", fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

# Row 2: Gray Matter
axes[1, 0].imshow(im2, cmap='bone')
axes[1, 0].set_title("Original Brain (Bone Colormap)", fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

y_gm = np.interp(x, bp_gray_matter[:, 0], bp_gray_matter[:, 1])
axes[1, 1].plot(x, y_gm, 'g-', linewidth=2.5, label="GM Transform")
axes[1, 1].scatter(bp_gray_matter[:, 0], bp_gray_matter[:, 1], color='black', s=40, zorder=5)
axes[1, 1].plot([0, 255], [0, 255], 'k--', alpha=0.3, label="Identity")
axes[1, 1].set_title("Gray Matter Transformation Curve", fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel("Input Intensity (r)")
axes[1, 1].set_ylabel("Output Intensity (s)")
axes[1, 1].set_xlim([0, 255])
axes[1, 1].set_ylim([0, 255])
axes[1, 1].grid(True, linestyle=':', alpha=0.6)
axes[1, 1].legend()

axes[1, 2].imshow(im2_gm, cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title("Gray Matter Accentuated", fontsize=12, fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

### Results and Interpretation for Question 2
- **White Matter Accentuation:** The high slope in $[140, 220]$ successfully amplifies the central white matter tracts (e.g. corpus callosum and internal capsule) against the surrounding cortex.
- **Gray Matter Accentuation:** The bandpass curve selectively isolates the outer cerebral cortex and basal ganglia structures while simultaneously dimming both the darker cerebrospinal fluid (ventricles) and the brighter white matter tracts.

---
# Question 3: Gamma Correction on L* Plane in L*a*b* Color Space

### Problem Statement
Consider the image shown in Fig. 3:
- (a) Apply gamma correction to the $L$ plane in the $L^*a^*b^*$ color space and state the $\gamma$ value.
- (b) Show the histograms of the original and corrected images.

### Theory & Formulation
Standard gamma correction on individual RGB channels often results in color shifts and unnatural saturation changes because the chromatic ratios are distorted.
In contrast, the CIELAB ($L^*a^*b^*$) color space explicitly decouples lightness ($L^*$) from color opponents ($a^*$, $b^*$):
- $L^*$ corresponds to perceptual Lightness ($0$ to $100$, scaled to $[0, 255]$ in 8-bit OpenCV).
- $a^*$ represents the Green-Red color axis.
- $b^*$ represents the Blue-Yellow color axis.

**Gamma Power-Law Model on $L$:**
$$L_{out} = \text{round}\left( 255 \times \left( \frac{L_{in}}{255} \right)^\gamma \right)$$

For high-contrast images with heavy shadows, setting $\mathbf{\gamma < 1.0}$ (e.g., $\gamma = 0.65$) raises the values of low-intensity pixels to recover hidden shadow details while maintaining hue and saturation stability.

In [ ]:
def gamma_correction_lab(im_bgr: np.ndarray, gamma: float = 0.65):
    """
    Applies gamma correction exclusively to the L plane in LAB color space.
    """
    # 1. Convert BGR -> LAB
    lab = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2LAB)
    l_plane, a_plane, b_plane = cv2.split(lab)
    
    # 2. Build Gamma LUT for L plane
    lut = np.array([
        np.clip(np.round(255.0 * ((i / 255.0) ** gamma)), 0, 255)
        for i in range(256)
    ], dtype=np.uint8)
    
    # 3. Apply LUT on L plane and recombine
    l_corrected = cv2.LUT(l_plane, lut)
    lab_corrected = cv2.merge([l_corrected, a_plane, b_plane])
    
    # 4. Convert back to BGR
    corrected_bgr = cv2.cvtColor(lab_corrected, cv2.COLOR_LAB2BGR)
    
    return corrected_bgr, l_plane, l_corrected

In [ ]:
# Load Figure 3 or synthetic test color image
img3_path = 'fig3.png'
if os.path.exists(img3_path):
    im3 = cv2.imread(img3_path)
else:
    h, w = 400, 500
    x, y = np.meshgrid(np.linspace(0, 1, w), np.linspace(0, 1, h))
    synth = np.zeros((h, w, 3), dtype=np.uint8)
    synth[:, :, 0] = np.clip(180 + 50 * y, 0, 255).astype(np.uint8)
    synth[:, :, 1] = np.clip(140 + 40 * y, 0, 255).astype(np.uint8)
    synth[:, :, 2] = np.clip(100 + 30 * y, 0, 255).astype(np.uint8)
    shadow_mask = (y > 0.4) & (x > 0.2) & (x < 0.8)
    synth[shadow_mask] = (synth[shadow_mask] * 0.22).astype(np.uint8)
    im3 = synth

# Chosen Gamma Value (Stated: gamma = 0.65)
gamma_val = 0.65
im3_corrected, l_orig, l_corr = gamma_correction_lab(im3, gamma=gamma_val)

print(f"Stated Gamma Value: gamma = {gamma_val}")

# Visual and Histogram Comparison
im3_rgb = cv2.cvtColor(im3, cv2.COLOR_BGR2RGB)
im3_corr_rgb = cv2.cvtColor(im3_corrected, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-Left: Original Image
axes[0, 0].imshow(im3_rgb)
axes[0, 0].set_title("Original Image (Fig. 3)", fontsize=13, fontweight='bold')
axes[0, 0].axis('off')

# Top-Right: Corrected Image
axes[0, 1].imshow(im3_corr_rgb)
axes[0, 1].set_title(f"Gamma Corrected Image (L* Plane, γ = {gamma_val})", fontsize=13, fontweight='bold')
axes[0, 1].axis('off')

# Bottom-Left: Original L Histogram
hist_orig = cv2.calcHist([l_orig], [0], None, [256], [0, 256])
axes[1, 0].plot(hist_orig, color='navy', linewidth=2, label='L* Histogram')
axes[1, 0].fill_between(range(256), hist_orig.ravel(), color='navy', alpha=0.25)
axes[1, 0].set_title("Histogram of Original L* Plane", fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel("Lightness L* (0 - 255)")
axes[1, 0].set_ylabel("Pixel Frequency")
axes[1, 0].set_xlim([0, 255])
axes[1, 0].grid(True, linestyle=':', alpha=0.6)
axes[1, 0].legend()

# Bottom-Right: Corrected L Histogram
hist_corr = cv2.calcHist([l_corr], [0], None, [256], [0, 256])
axes[1, 1].plot(hist_corr, color='darkred', linewidth=2, label=f'Corrected L* (γ = {gamma_val})')
axes[1, 1].fill_between(range(256), hist_corr.ravel(), color='darkred', alpha=0.25)
axes[1, 1].set_title(f"Histogram of Corrected L* Plane (γ = {gamma_val})", fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel("Lightness L* (0 - 255)")
axes[1, 1].set_ylabel("Pixel Frequency")
axes[1, 1].set_xlim([0, 255])
axes[1, 1].grid(True, linestyle=':', alpha=0.6)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### Results and Interpretation for Question 3
1. **Stated $\gamma$ Value:** $\mathbf{\gamma = 0.65}$
2. **Histogram Shift & Visual Impact:**
   - In the original histogram, a substantial concentration of dark pixels resided in the range $[0, 50]$, representing deep shadow regions on the stairs and clothing.
   - With $\gamma = 0.65$, this low-intensity peak is redistributed rightward into the mid-tone range $[70, 140]$, rendering previously obscured details clearly visible.
   - Crucially, because the operation was performed strictly on the $L^*$ plane without altering the $a^*$ and $b^*$ chrominance channels, no chromatic aberration, color cast, or tint shift was introduced.